In [1]:
from pathlib import Path

from katabatic.artifacts import LocalArtifactStore
from katabatic.models.tabddpm.models import Tabddpm
from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.utils.preprocess import preprocess_dataset

ROOT = None

for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / "datasets").exists() and (p / "models").exists():
        ROOT = p
        break

raw_file = ROOT / "datasets" / "car.csv"
processed_file = ROOT / "preprocessed_data" / "car_tabddpm.csv"
artifact_dir = ROOT / "artifacts"

print("ROOT:", ROOT)
print("Raw dataset:", raw_file)
print("Dataset exists:", raw_file.exists())

processed_file.parent.mkdir(parents=True, exist_ok=True)

preprocess_dataset(
    str(raw_file),
    str(processed_file)
)

store = LocalArtifactStore(str(artifact_dir))

pipeline = TrainTestSplitPipeline(
    model=Tabddpm()
)

pipeline._evaluations = []

results = pipeline.run(
    input_csv=str(processed_file),
    dataset_name="car_tabddpm",
    artifact_store=store,
    model_name="tabddpm",
)

print(results)

ROOT: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic
Raw dataset: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\datasets\car.csv
Dataset exists: True
Preprocessing: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\datasets\car.csv
Saved preprocessed dataset to: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\preprocessed_data\car_tabddpm.csv
Loaded data with shape: (1728, 7)
Train label distribution:
 6
unacc    0.700434
acc      0.222142
good     0.039797
vgood    0.037627
Name: proportion, dtype: float64
Test label distribution:
 6
unacc    0.699422
acc      0.222543
good     0.040462
vgood    0.037572
Name: proportion, dtype: float64
Saved dataset artifact under datasets/car_tabddpm/split-20260808-123525
Step 100/200 | MLoss: 1.2292 | GLoss: 0.0000
Step 200/200 | MLoss: 1.3281 | GLoss: 0.0000
{'message': 'Train test split pipeline executed successfull

In [2]:
import pandas as pd

from katabatic.pipeline.evaluation_pipeline import SyntheticEvaluationPipeline

splits_root = ROOT / "artifacts" / "datasets" / "car_tabddpm"

split_dirs = sorted(
    [p for p in splits_root.glob("split-*") if p.is_dir()],
    key=lambda p: p.stat().st_mtime
)

latest_split = split_dirs[-1]

print("Using split:", latest_split)

train_df = pd.read_csv(
    latest_split / "train" / "train_full.csv"
)

test_df = pd.read_csv(
    latest_split / "test" / "test_full.csv"
)

target_col = train_df.columns[-1]

categorical_cols = train_df.columns[:-1].tolist()
continuous_cols = []

model = pipeline.model

synthetic_df = model.sample(
    len(train_df),
    seed=42
)

print("Target column:", target_col)
print("Synthetic type:", type(synthetic_df))

print("\nReal columns:")
print(train_df.columns.tolist())

print("\nSynthetic columns:")
print(synthetic_df.columns.tolist())

print("\nSynthetic sample:")
print(synthetic_df.head())

evaluation_pipeline = SyntheticEvaluationPipeline(
    dimensions=[
        "fidelity",
        "utility",
        "diversity",
        "privacy",
        "consistency",
        "stability",
    ],
    categorical_cols=categorical_cols,
    continuous_cols=continuous_cols,
)

report = evaluation_pipeline.run(
    real_data=train_df,
    synthetic_data=synthetic_df,
    target_col=target_col,
    test_data=test_df,
    model=model,
)

print("\nComposite Score:", report.composite_score)

print("\nDimension Scores:")
for dimension, score in report.dimension_scores.items():
    print(f"{dimension}: {score}")

Using split: c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\katabatic\artifacts\datasets\car_tabddpm\split-20260808-123525
Target column: 6
Synthetic type: <class 'pandas.core.frame.DataFrame'>

Real columns:
['0', '1', '2', '3', '4', '5', '6']

Synthetic columns:
['0', '1', '2', '3', '4', '5', '6']

Synthetic sample:
      0     1  2  3    4     5      6
0  high  high  2  2  big  high    acc
1  high  high  2  2  big  high    acc
2  high  high  2  2  big  high    acc
3  high  high  2  2  big  high    acc
4  high  high  2  2  big  high  unacc

Running fidelity evaluation...

=== Fidelity Evaluation ===
Overall fidelity score: 0.4120

Categorical JSD (lower = better)  ->  score: 0.4120
  0                              JSD = 0.6104
  1                              JSD = 0.6179
  2                              JSD = 0.6104
  3                              JSD = 0.5667
  4                              JSD = 0.5627
  5                              JSD = 0.5600
  avg  

c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:780: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\savin\OneDrive\Desktop\Semester 02\Capstone Project\Katabatic\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:780:


=== Utility Evaluation ===
Overall utility score: 0.7927

Classifier   Metric     TSTR mean    TRTR mean    Delta   
--------------------------------------------------------
LR           accuracy   0.6994       0.6884       -0.011
LR           f1         0.5757       0.6097       0.034
DT           accuracy   0.6994       0.9734       0.274
DT           f1         0.5757       0.9730       0.3973
RF           accuracy   0.6994       0.9671       0.2677
RF           f1         0.5757       0.9668       0.3911
LinearSVM    accuracy   0.6994       0.7023       0.0029
LinearSVM    f1         0.5757       0.6239       0.0482
MLP          accuracy   0.6994       0.9717       0.2723
MLP          f1         0.5757       0.9717       0.396

Running diversity evaluation...

=== Diversity Evaluation ===
Overall diversity score: 0.2911

Category Coverage (% of real categories in synth)
  0                              25.0%
  1                              25.0%
  2                              2